# 05B PHA parameters with CHARMM-GUI (CGenFF)

**What this does:** checks the PHA force-field files made on the CHARMM-GUI website.
**You need:** the PHA SDF file from notebook 04 and your CHARMM-GUI download.
**You get:** checked files to use in notebook 06B.

In [ ]:
from pathlib import Path
import sys

repo_root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "iphasimulator").exists())
if str(repo_root / "src") not in sys.path:
    sys.path.insert(0, str(repo_root / "src"))

from rdkit import Chem
from iphasimulator.charmmgui_import import PENALTY_GRADE_RULES, penalty_grade, read_cgenff_penalties

EXAMPLE_DIR = repo_root / "examples" / "data" / "charmm_gui_ANC55_P3HB4"
print("✓ Ready")

## Step 1: On the CHARMM-GUI website

1. Go to [https://www.charmm-gui.org](https://www.charmm-gui.org) and open **Ligand Reader & Modeler**.
2. Upload your PHA SDF file from notebook 04, e.g. `PHB4_R.sdf`.
3. Click **Next** until the last page, then download the result.
4. Unzip it: the `lig/` folder is what you need.

## Step 2: Point to your download

Set `LIG_DIR` to your `lig/` folder and `SDF_PATH` to the SDF you uploaded. Without changes, the example data is used.

In [ ]:
LIG_DIR = EXAMPLE_DIR / "lig"            # your lig/ folder
SDF_PATH = EXAMPLE_DIR / "PHB4_R.sdf"    # the SDF you uploaded

for path in (LIG_DIR / "lig_g.rtf", LIG_DIR / "lig.prm", LIG_DIR / "lig.rtf", SDF_PATH):
    if path.is_file():
        print(f"✓ {path.name}")
    elif path.name == "lig_g.rtf":
        print("✗ lig_g.rtf is not in this download; use lig.rtf instead")
    else:
        print(f"✗ {path.name} not found in {path.parent}")

## Step 3: Is the PHA still all-R?

Every stereocentre in the PHA should be R. This reads them from the SDF you uploaded.

In [ ]:
mol = Chem.MolFromMolFile(str(SDF_PATH), removeHs=False)
Chem.AssignStereochemistryFrom3D(mol)
labels = [label for _, label in Chem.FindMolChiralCenters(mol, includeUnassigned=True, useLegacyImplementation=False)]

all_r = bool(labels) and labels.count("R") == len(labels)
print(f"{labels.count('R')} of {len(labels)} stereocentres are R {'✓' if all_r else '✗'}")

## Step 4: Are the parameters reliable?

CHARMM-GUI gives each set of parameters a quality score: lower is better. Below 10 is GOOD, 10–50 needs a CHECK, above 50 is POOR.

In [ ]:
report = read_cgenff_penalties(LIG_DIR)
marks = {"GOOD": "✓", "CHECK": "!", "POOR": "✗"}

for title, score in (("Parameter", report.max_parameter_penalty), ("Charge", report.max_charge_penalty)):
    grade = penalty_grade(score)
    print(f"{title} quality score: {score:g} → {grade} ({PENALTY_GRADE_RULES[grade]}) {marks[grade]}")
print(f"CGenFF version: program {report.program_version}, parameter files {report.parameter_files_version}")

## Next

Open 06B. In CHARMM-GUI Solution Builder, upload `lig_g.rtf` (topology) and `lig.prm` (parameters).
CGenFF parameters for PHA are estimated from similar small molecules; keep the quality scores with your results.